# Task 2 — NER (`LOCATION`, `ORGANIZATION`, `PERSON`) & LDA Topic Modeling (Google Colab Runner)

Notebook ini menjalankan seluruh pipeline **Task 2 (T2)** sesuai rancangan:
- **Phase 1 & 2 · E1 CPU Baseline:** Feature-based BIO2 + Cased Civic Gazetteer & 3-Gate `, ` Post-Processing
- **Phase 3 · LDA Topic Modeling:** 4-Layer Stopwords + Gensim `Phrases` Bigrams + Sweep `K in {5, 8, 10, 12, 15}` (fit ketat pada `split == "train"`)
- **Phase 4 · Transformer NER (E2–E4):** Fine-tuning **IndoBERT** (`indobenchmark/indobert-base-p1`, `max_length=384`, dynamic padding, `--fp16` + `GradScaler` guard) untuk:
  - `ner_full` (Real + Synthetic, 3 seed: `42, 123, 2025`) $\rightarrow$ **E2** (Raw) & **E3** (Main: + Civic Post-Processing)
  - `ner_real_only` (Real-Only Ablation, 3 seed: `42, 123, 2025`) $\rightarrow$ **E4a** (Raw) & **E4b** (+ Civic Post-Processing)
- **Phase 5 · Verifikasi & Packaging:** Unit test + inferensi `T2Predictor` + arsip scoped `outputs/task2_*.json` & `models/task2/`

In [1]:
!nvidia-smi

Sun Nov 22 14:52:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from getpass import getpass

PROJECT_DIR = "/content/Tubes_NLP"
if not os.path.exists(PROJECT_DIR):
    token = getpass("Masukkan GitHub Token: ")
    REPO_URL = f"https://{token}@github.com/RylRizqi16/Tubes_NLP.git"
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

Masukkan GitHub Token: ··········
Cloning into '/content/Tubes_NLP'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 150 (delta 40), reused 125 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 14.89 MiB | 19.21 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/Tubes_NLP


In [3]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [4]:
!pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.6.5 which is incompatible.


In [5]:
from pathlib import Path

master_path = Path("data/processed/master_dataset.jsonl")
if not master_path.exists():
    print("master_dataset.jsonl belum ada, mengunduh & membangun dataset...")
    !python scripts/download_datasets.py
    !python scripts/generate_synthetic_edge_cases.py
    !python scripts/build_master_dataset.py
else:
    print(f"Dataset siap: {master_path} ({master_path.stat().st_size / 1e6:.2f} MB)")

Dataset siap: data/processed/master_dataset.jsonl (8.90 MB)


In [6]:
!python -m src.task2.train_ner_baseline

=== PHASE 1 · SPAN ALIGNMENT CHECK (max_length=384) ===
  Records=9,257 | Gold Spans=21,002 | Recovered=21,002 | Truncated=0 | Loss=0.000000

=== PHASE 2 · 3-GATE COMMA MERGER & POST-PROCESSING SAFEGUARD (VAL) ===
  Oracle Upper Bound (gold -> post): all=1.0000 | real=1.0000 | synthetic=1.0000
  val (real)      Micro-F1: raw=0.7134 -> post=0.7152 (delta=+0.0018, no_regression=True)
  val (synthetic) Micro-F1: raw=0.6778 -> post=0.9948 (delta=+0.3170)

=== E1 BASELINE (FEATURE BIO2 + CIVIC GAZETTEER) ===

[val_metrics] latency=5.0538 ms/sample
  all              docs=1388 spans=3198 | P=0.8251 R=0.7658 Micro-F1=0.7944 Macro-F1=0.7945 | LOC=0.7905 ORG=0.7959 PER=0.7972
  real             docs= 974 spans=2327 | P=0.7550 R=0.6794 Micro-F1=0.7152 Macro-F1=0.7113 | LOC=0.6849 ORG=0.6785 PER=0.7706
  synthetic        docs= 414 spans= 871 | P=0.9931 R=0.9966 Micro-F1=0.9948 Macro-F1=0.9961 | LOC=0.9922 ORG=0.9961 PER=1.0000
  indonlu_nergrit  docs= 313 spans= 982 | P=0.6574 R=0.5509 Micro-F1=0

In [7]:
!python -m src.task2.train_lda --backend gensim --k-values 5 8 10 12 15 --seed 42

=== TASK 2 PHASE 3 · LDA TOPIC MODELING (backend=gensim, train=6,468, vocab=1,342) ===
| K | C_v Coherence | U_mass | Topic Diversity | Val Perplexity | Test Perplexity | Domain Purity | NMI |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| 5 | 0.3665 | -11.9873 | 0.9400 | 1200.65 | 1145.34 | 0.5806 | 0.2059 |
| 8 | 0.3943 | -11.4715 | 0.9500 | 1365.45 | 1307.37 | 0.6712 | 0.3106 |
| 10 | 0.4106 | -12.4201 | 0.9700 | 1329.66 | 1280.26 | 0.6027 | 0.2079 |
| 12 | 0.3654 | -12.4309 | 0.9417 | 1416.51 | 1386.01 | 0.7292 | 0.3124 |
| 15 * | 0.4391 | -12.9338 | 0.9267 | 1619.67 | 1630.14 | 0.6674 | 0.2412 |

Selected K=15 Top Keywords:
  topic_0  | dom=Layanan & Administrasi (0.72) | src={'sp4n_lapor': 21, 'campus_complaints': 139, 'synthetic': 94} | error, pengumuman, dokter_poli, anak_sesuai, sistem_antrean, online_dukcapil, perihal, asli_bad
  topic_1  | dom=Layanan & Administrasi (0.85) | src={'sp4n_lapor': 454, 'campus_complaints': 347, 'synthetic': 0} | aplikasi, ul

In [8]:
!python -m src.task2.train_ner_transformer \
    --model-name indobenchmark/indobert-base-p1 \
    --seeds 42 123 2025 \
    --epochs 5 \
    --batch-size 16 \
    --lr 3e-5 \
    --max-length 384 \
    --patience 2 \
    --fp16

Loaded Task 2 NER (ner_full) -> train=6,479 (subset=all), val=1,388, test=1,390 | model=indobenchmark/indobert-base-p1, max_length=384, use_crf=False, fp16=True
config.json: 100% 1.53k/1.53k [00:00<00:00, 488kB/s]
tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 9.99kB/s]
vocab.txt: 100% 229k/229k [00:00<00:00, 79.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 571kB/s]

pytorch_model.bin: downloading bytes:  55% 273M/498M [00:02<00:00, 291MB/s, 20.5MB/s  ]
pytorch_model.bin: downloading bytes:  95% 472M/498M [00:02<00:00, 387MB/s, 41.2MB/s  ]
pytorch_model.bin: reconstructing file:  81% 402M/498M [00:02<00:00, 190MB/s, 12.7MB/s  ]
pytorch_model.bin: downloading bytes: 100% 472M/472M [00:03<00:00, 153MB/s, 43.0MB/s  ]
pytorch_model.bin: reconstructing file: 100% 498M/498M [00:03<00:00, 161MB/s, 45.8MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 18475.07it/s]

model.safetensors: downloading bytes:  73% 365M/498M [00:01<00:00, 463MB/s, 23.6MB/s  ]
model.safetensors: do

In [9]:
!python -m src.task2.train_ner_transformer \
    --model-name indobenchmark/indobert-base-p1 \
    --seeds 42 123 2025 \
    --epochs 5 \
    --batch-size 16 \
    --lr 3e-5 \
    --max-length 384 \
    --patience 2 \
    --fp16 \
    --real-only

Loaded Task 2 NER (ner_real_only) -> train=4,547 (subset=real), val=1,388, test=1,390 | model=indobenchmark/indobert-base-p1, max_length=384, use_crf=False, fp16=True
Loading weights: 100% 199/199 [00:00<00:00, 13316.10it/s]
[ner_real_only seed=42 epoch=1/5] {'epoch': 1, 'train_loss': 0.3454, 'val_micro_f1': 0.6268, 'val_macro_f1': 0.6391, 'val_partial_f1': 0.845, 'latency_ms_per_sample': 0.532}
[ner_real_only seed=42 epoch=2/5] {'epoch': 2, 'train_loss': 0.0988, 'val_micro_f1': 0.6119, 'val_macro_f1': 0.6268, 'val_partial_f1': 0.8392, 'latency_ms_per_sample': 0.5346}
[ner_real_only seed=42 epoch=3/5] {'epoch': 3, 'train_loss': 0.0512, 'val_micro_f1': 0.6249, 'val_macro_f1': 0.6372, 'val_partial_f1': 0.8464, 'latency_ms_per_sample': 0.5078}
[ner_real_only seed=42] Early stopping at epoch 3.
Loading weights: 100% 199/199 [00:00<00:00, 13385.07it/s]
[ner_real_only seed=123 epoch=1/5] {'epoch': 1, 'train_loss': 0.3545, 'val_micro_f1': 0.6046, 'val_macro_f1': 0.6161, 'val_partial_f1': 0.81

In [10]:
!python -m src.task2.evaluate_ner --split test


=== TASK 2 NER EVALUATION SUMMARY (TEST · STRICT SPAN MICRO-F1 / MACRO-F1) ===
| Config | All (Micro / Macro F1) | Real (Micro / Macro F1) | Synthetic (Micro / Macro F1) | Partial-Boundary F1 (All / Syn) |
| :--- | :---: | :---: | :---: | :---: |
| E1_Baseline_Civic_BIO | 0.7900 / 0.7903 | 0.7102 / 0.7070 | 1.0000 / 1.0000 | 0.8554 / 1.0000 |
| E2_IndoBERT_Full_Raw | 0.8619 ± 0.0029 / 0.8643 ± 0.0034 | 0.8299 ± 0.0055 / 0.8262 ± 0.0066 | 0.9505 ± 0.0045 / 0.9546 ± 0.0032 | 0.9195 ± 0.0019 / 0.9914 ± 0.0033 |
| E3_IndoBERT_Full_Civic_Hybrid | 0.8794 ± 0.0032 / 0.8814 ± 0.0038 | 0.8361 ± 0.0043 / 0.8327 ± 0.0052 | 0.9996 ± 0.0007 / 0.9997 ± 0.0005 | 0.9220 ± 0.0019 / 0.9996 ± 0.0007 |
| E4a_Ablation_RealOnly_Raw | 0.6122 ± 0.0135 / 0.6376 ± 0.0154 | 0.8191 ± 0.0212 / 0.8158 ± 0.0216 | 0.1361 ± 0.0105 / 0.1283 ± 0.0188 | 0.8431 ± 0.0117 / 0.7479 ± 0.0209 |
| E4b_Ablation_RealOnly_Civic_Hybrid | 0.8599 ± 0.0157 / 0.8632 ± 0.0157 | 0.8265 ± 0.0191 / 0.8232 ± 0.0195 | 0.9519 ± 0.0197 / 0.96

In [11]:
!python -m unittest src.task2.test_task2 -v

from src.task2.predictor import T2Predictor

t2 = T2Predictor()
print(f"Active NER backend: {t2.ner_backend} (checkpoint={t2.checkpoint_dir})")
samples = [
    "Lampu jalan di Jl. Setiabudi, Srondol mati 5 hari, mohon Dishub Kota Semarang segera perbaiki.",
    "Terima kasih Pak Budi dan Puskesmas Tembalang atas pelayanan cepat hari ini.",
    "Aplikasi SP4N LAPOR sering error saat upload foto laporan warga.",
]
for text in samples:
    print("\nTEXT:", text)
    print("  NER  :", t2.predict(text))
    topic = t2.predict_topic(text)
    print(f"  TOPIC: topic_{topic['topic_id']} (p={topic['topic_probability']:.4f}) -> {topic['top_keywords'][:6]}")

test_adjacent_same_label_b_spans_preserved_across_single_space (src.task2.test_task2.Task2AlignmentAndDecodingTest.test_adjacent_same_label_b_spans_preserved_across_single_space) ... ok
test_master_dataset_zero_truncation_and_roundtrip_alignment (src.task2.test_task2.Task2AlignmentAndDecodingTest.test_master_dataset_zero_truncation_and_roundtrip_alignment) ... ok
test_subword_continuation_repair_vs_orphan_i_opt_in (src.task2.test_task2.Task2AlignmentAndDecodingTest.test_subword_continuation_repair_vs_orphan_i_opt_in) ... ok
test_kantor_gating_and_strict_expansion_only (src.task2.test_task2.Task2PostProcessingAndOracleTest.test_kantor_gating_and_strict_expansion_only) ... ok
test_oracle_postprocessing_upper_bound_on_val (src.task2.test_task2.Task2PostProcessingAndOracleTest.test_oracle_postprocessing_upper_bound_on_val) ... ok
test_three_gate_comma_merger_and_guards (src.task2.test_task2.Task2PostProcessingAndOracleTest.test_three_gate_comma_merger_and_guards) ... ok
test_lda_preprocess

  TOPIC: topic_10 (p=0.5344) -> ['darurat', 'satgas_ppks', 'sarpras', 'gedung', 'perlindungan', 'ppks']

TEXT: Terima kasih Pak Budi dan Puskesmas Tembalang atas pelayanan cepat hari ini.
  NER  : T2Result(entities=[EntitySpan(text='Pak Budi', label='PERSON', start=13, end=21), EntitySpan(text='Puskesmas Tembalang', label='ORGANIZATION', start=26, end=45)])
  TOPIC: topic_4 (p=0.6511) -> ['pupr', 'pln', 'dishub', 'dlh', 'bpbd', 'rsud']

TEXT: Aplikasi SP4N LAPOR sering error saat upload foto laporan warga.
  NER  : T2Result(entities=[])
  TOPIC: topic_1 (p=0.2761) -> ['aplikasi', 'ult', 'gedung', 'loket', 'operasional', 'staf']


In [12]:
!zip -rq task2_colab_results.zip outputs/task2_*.json models/task2/
print("Siap diunduh: task2_colab_results.zip")

Siap diunduh: task2_colab_results.zip
